# 01 — Data Collection

**Purpose**: Collect stock price data from Yahoo Finance and CLOB prices from Polymarket API.

**Outputs**:
- `data/stock_prices_daily.parquet` — OHLCV for AAPL, NVDA, AMZN (+ all tickers with markets)
- `data/clob_prices_*.parquet` — recent CLOB price snapshots (where available)

In [ ]:
import time
from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf
import requests

import config as cfg
import thesis_utils as tu

data_dir, results_dir = tu.ensure_project_dirs(cfg.PROJECT_ROOT)
print(f'Data dir: {data_dir}')

## 1. Yahoo Finance — Stock Prices

Download daily OHLCV for all tickers that appear in the market dataset.
We need:
- Close prices for resolution matching (all market types)
- High prices for monthly_hit markets
- Open prices for daily_updown (previous close comparison)

In [ ]:
# Load the full market inventory to find all tickers
inv_path = results_dir / 'full_market_inventory.csv'
if inv_path.exists():
    df_inv = pd.read_csv(inv_path)
    all_tickers = sorted(df_inv['ticker'].dropna().unique())
else:
    # Fallback: use primary tickers + known extras
    all_tickers = cfg.ALL_TICKERS

# Add SPY for tariff/market context
tickers_to_download = list(set(all_tickers) | {'SPY'})
print(f'Tickers to download: {tickers_to_download}')
print(f'Date range: {cfg.DATA_START} to {cfg.DATA_END}')

In [ ]:
# Download OHLCV data from Yahoo Finance
frames = {}
failed = []

for ticker in tickers_to_download:
    try:
        t = yf.Ticker(ticker)
        hist = t.history(start=cfg.DATA_START, end=cfg.DATA_END, auto_adjust=False)
        if len(hist) == 0:
            failed.append((ticker, 'no data'))
            continue
        # Rename columns to include ticker prefix
        renamed = hist[['Open', 'High', 'Low', 'Close', 'Volume']].rename(
            columns={
                'Open': f'{ticker}_Open',
                'High': f'{ticker}_High',
                'Low': f'{ticker}_Low',
                'Close': f'{ticker}_Close',
                'Volume': f'{ticker}_Volume',
            }
        )
        frames[ticker] = renamed
        print(f'  {ticker}: {len(hist):,} days ({hist.index.min().date()} to {hist.index.max().date()})')
    except Exception as e:
        failed.append((ticker, str(e)))
        print(f'  {ticker}: FAILED — {e}')
    time.sleep(0.5)  # Rate limit courtesy

if failed:
    print(f'\nFailed tickers: {failed}')

In [ ]:
# Merge all tickers into a single DataFrame
if frames:
    stock_prices = pd.concat(frames.values(), axis=1)
    stock_prices.index = pd.to_datetime(stock_prices.index).tz_localize(None)
    stock_prices.index.name = 'date'

    # Save
    out_path = data_dir / 'stock_prices_daily.parquet'
    tu.save_parquet(stock_prices, out_path)
    print(f'Saved {stock_prices.shape} to {out_path}')
    print(f'Date range: {stock_prices.index.min().date()} to {stock_prices.index.max().date()}')
    print(f'Tickers: {sorted(set(c.split("_")[0] for c in stock_prices.columns))}')
else:
    print('No data downloaded!')

## 2. Polymarket CLOB API — Recent Market Prices

The CLOB API provides current/recent order book prices. For markets closed in the last ~30 days,
we can potentially retrieve the final trading price (implied probability).

In [ ]:
# Load filtered markets with token IDs
df_mkts = tu.load_parquet('polymarket_stocks_markets.parquet')
print(f'Markets: {len(df_mkts):,}')
print(f'Closed: {df_mkts["closed"].sum():,}')

# Parse token IDs
df_mkts['token_list'] = df_mkts['clobTokenIds'].apply(tu.parse_token_ids)
df_mkts['n_tokens'] = df_mkts['token_list'].apply(len)
print(f'Markets with valid token IDs: {(df_mkts["n_tokens"] > 0).sum():,}')

In [ ]:
def fetch_clob_price(token_id: str) -> dict | None:
    """Fetch last trade price from CLOB API for a token."""
    url = f'{cfg.CLOB_BASE}/price'
    params = {'token_id': token_id, 'side': 'buy'}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

# Sample a few markets to test API availability
sample_tokens = df_mkts[df_mkts['n_tokens'] > 0].head(5)
for _, row in sample_tokens.iterrows():
    tokens = row['token_list']
    if tokens:
        result = fetch_clob_price(tokens[0])
        q_short = row['question'][:60]
        print(f'  {q_short}... -> {result}')
        time.sleep(0.3)

In [ ]:
# Batch fetch CLOB prices for closed markets (Yes token = first token ID)
# Only fetch for markets with valid tokens — rate limit to avoid API issues
clob_rows = []
fetch_markets = df_mkts[df_mkts['n_tokens'] > 0].copy()
total = len(fetch_markets)
errors = 0

for i, (_, row) in enumerate(fetch_markets.iterrows()):
    if i % 200 == 0:
        print(f'  Fetching {i}/{total}...')
    token_yes = row['token_list'][0]  # First token = Yes outcome
    result = fetch_clob_price(token_yes)
    if result and 'price' in result:
        clob_rows.append({
            'market_id': row['market_id'],
            'clob_price': float(result['price']),
        })
    else:
        errors += 1
        clob_rows.append({
            'market_id': row['market_id'],
            'clob_price': np.nan,
        })
    time.sleep(0.15)  # ~6 req/sec

df_clob = pd.DataFrame(clob_rows)
valid_prices = df_clob['clob_price'].notna().sum()
print(f'\nFetched {valid_prices:,} valid CLOB prices out of {total:,} markets ({errors} errors)')

In [ ]:
# Save CLOB prices
if len(df_clob) > 0:
    clob_path = data_dir / 'clob_prices_snapshot.parquet'
    tu.save_parquet(df_clob, clob_path)
    print(f'Saved {len(df_clob):,} CLOB prices to {clob_path}')
    print(f'\nCLOB price distribution (valid only):')
    print(df_clob['clob_price'].dropna().describe())

## 3. Verification

In [ ]:
# Verify stock prices loaded correctly
sp = tu.load_parquet('stock_prices_daily.parquet', required=False)
if sp is not None:
    print('Stock prices:')
    print(f'  Shape: {sp.shape}')
    print(f'  Date range: {sp.index.min()} to {sp.index.max()}')
    # Check for primary tickers
    for tk in cfg.PRIMARY_TICKERS:
        col = f'{tk}_Close'
        if col in sp.columns:
            print(f'  {tk}: {sp[col].notna().sum()} trading days, last close = ${sp[col].dropna().iloc[-1]:.2f}')
        else:
            print(f'  {tk}: MISSING')

# Verify CLOB prices
cp = tu.load_parquet('clob_prices_snapshot.parquet', required=False)
if cp is not None:
    print(f'\nCLOB prices: {cp["clob_price"].notna().sum():,} valid / {len(cp):,} total')
    print(f'  Note: Many closed markets return NaN (purged from CLOB after ~30 days)')